# OPDI Flights cleaning

In [14]:
from pyspark.sql.functions import col, count, when

df_raw = spark.table("bronze.opdi_flight_list")

print(f"Total raw rows : {df_raw.count()}")
print(f"Total columns  : {len(df_raw.columns)}")

# Null counts on key columns including backup candidates
display(
    df_raw.select([
        count(when(col(c).isNull(), 1)).alias(c)
        for c in ["id", "flt_id", "dof", "unix_time",
                  "adep", "adep_p", "ades", "ades_p", "icao_operator"]
    ])
)

# Check combined null coverage after coalesce
print(f"adep AND adep_p both NULL : {df_raw.filter(col('adep').isNull() & col('adep_p').isNull()).count()}")
print(f"ades AND ades_p both NULL : {df_raw.filter(col('ades').isNull() & col('ades_p').isNull()).count()}")
print(f"dof AND unix_time both NULL : {df_raw.filter(col('dof').isNull() & col('unix_time').isNull()).count()}")
print(f"id NULL : {df_raw.filter(col('id').isNull()).count()}")

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 17, Finished, Available, Finished, False)

Total raw rows : 43540769
Total columns  : 20


SynapseWidget(Synapse.DataFrame, d961f9b6-c084-4880-81c1-f1f2ae1f6dcc)

adep AND adep_p both NULL : 11807638


ades AND ades_p both NULL : 11807638
dof AND unix_time both NULL : 0
id NULL : 0


In [15]:
from pyspark.sql.functions import col, date_format, trim, upper, coalesce, lit 

df_clean = (
    df_raw
    .select(
        col("id").cast("long").alias("flight_id"),      # never NULL — no backup needed
        col("dof").cast("date").alias("flight_date"),   # never NULL — no backup needed
        trim(upper(coalesce(col("adep"), col("adep_p")))).alias("adep_icao"),  # fallback saves ~9.5M rows
        trim(upper(coalesce(col("ades"), col("ades_p")))).alias("ades_icao"),  # fallback saves ~8.1M rows
        trim(upper(col("icao_operator"))).alias("icao_operator"),
        col("typecode").alias("aircraft_type"),
        col("source_name"),
        col("batch_id"),
        col("loaded_at")
    )
    .withColumn("year_month", date_format(col("flight_date"), "yyyyMM").cast("integer"))
)

display(df_clean.limit(5))

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 29e15c06-bde2-48df-95e8-14a68472c8a6)

In [16]:
df_clean.printSchema()

# Null counts after transformation — compare with Cell 1 baseline
display(
    df_clean.select([
        count(when(col(c).isNull(), 1)).alias(c)
        for c in ["flight_id", "flight_date", "adep_icao", "ades_icao", "icao_operator"]
    ])
)

# year_month coverage — must span 2022-2024
display(df_clean.groupBy("year_month").count().orderBy("year_month"))

# Volume must match raw (no rows dropped at this stage)
clean_count = df_clean.count()
raw_count   = df_raw.count()
print(f"Raw   : {raw_count}")
print(f"Clean : {clean_count}")
print(f"Delta : {raw_count - clean_count} ({'OK — no rows dropped' if raw_count == clean_count else 'ERROR — investigate'})")

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 19, Finished, Available, Finished, False)

root
 |-- flight_id: long (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- adep_icao: string (nullable = true)
 |-- ades_icao: string (nullable = true)
 |-- icao_operator: string (nullable = true)
 |-- aircraft_type: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- loaded_at: string (nullable = true)
 |-- year_month: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, 8671faad-fbb4-409e-b1d9-8a82eae60377)

SynapseWidget(Synapse.DataFrame, 0aa077f1-3aaa-4b2d-8ff6-7d047704d114)

Raw   : 43540769
Clean : 43540769
Delta : 0 (OK — no rows dropped)


In [17]:
# Silver rejection: flight has no spatial anchor at all
df_null_rejected = (
    df_clean
    .filter(col("adep_icao").isNull() & col("ades_icao").isNull())
    .withColumn("rejection_reason", lit("null_adep_and_ades"))
)

df_pre_valid = df_clean.filter(
    col("adep_icao").isNotNull() | col("ades_icao").isNotNull()
)

raw_count   = df_raw.count()
valid_count = df_pre_valid.count()
rej_count   = df_null_rejected.count()

print(f"Raw           : {raw_count}")
print(f"Pre-valid     : {valid_count}  ({round(valid_count/raw_count*100, 1)}%)")
print(f"Null-rejected : {rej_count}  ({round(rej_count/raw_count*100, 1)}%)")
print(f"Control total : {valid_count + rej_count} ({'OK' if valid_count + rej_count == raw_count else 'ERROR'})")

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 20, Finished, Available, Finished, False)

Raw           : 43540769
Pre-valid     : 31733131  (72.9%)
Null-rejected : 11807638  (27.1%)
Control total : 43540769 (OK)


In [18]:
# Count duplicate flight_ids before dedup
dup_ids = df_pre_valid.groupBy("flight_id").count().filter(col("count") > 1)
print(f"flight_ids with duplicates : {dup_ids.count()}")
print(f"Total duplicate rows       : {dup_ids.agg({'count': 'sum'}).collect()[0][0]}")

display(dup_ids.orderBy("count", ascending=False).limit(10))

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 21, Finished, Available, Finished, False)

flight_ids with duplicates : 0


Total duplicate rows       : None


SynapseWidget(Synapse.DataFrame, 511d0979-b06b-487a-ae14-1334ac19110b)

In [21]:
from pyspark.sql.functions import when

df_airline = (
    spark.table("silver.airline")
    .select(col("icao_code").alias("icao_operator"), col("carrier_type"))
)

df_valid = (
    df_pre_valid
    .join(df_airline, on="icao_operator", how="left")
    .withColumn("lcc_flag",      when(col("carrier_type") == "LCC",      1).otherwise(0))
    .withColumn("network_flag",  when(col("carrier_type") == "Network",  1).otherwise(0))
    .withColumn("regional_flag", when(col("carrier_type") == "Regional", 1).otherwise(0))
    .withColumn("other_flag",    when(col("carrier_type") == "Other",    1).otherwise(0))
)

# Verify all flags — sum must equal total rows (every row belongs to exactly one category or NULL)
display(df_valid.groupBy("carrier_type").count().orderBy("count", ascending=False))

total = df_valid.count()
flag_sum = df_valid.selectExpr(
    "SUM(lcc_flag) as lcc",
    "SUM(network_flag) as network",
    "SUM(regional_flag) as regional",
    "SUM(other_flag) as other"
).collect()[0]
print(f"Total rows     : {total}")
print(f"lcc + network + regional + other : {flag_sum['lcc'] + flag_sum['network'] + flag_sum['regional'] + flag_sum['other']}")
print(f"Unclassified (carrier_type NULL) : {total - (flag_sum['lcc'] + flag_sum['network'] + flag_sum['regional'] + flag_sum['other'])}")

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8573779a-172c-43fd-94a0-4dba3e4d5aaa)

Total rows     : 31733131
lcc + network + regional + other : 15618253
Unclassified (carrier_type NULL) : 16114878


In [20]:
# Diagnose: find duplicate icao_codes in silver.airline
df_airline_check = spark.table("silver.airline")
display(
    df_airline_check.groupBy("icao_code")
    .count()
    .filter(col("count") > 1)
    .orderBy("count", ascending=False)
)

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 29d77379-ae06-400c-8839-9f6b55c51280)

In [22]:
# Write valid flights
df_valid.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.opdi_flights")

# Write rejected flights (null adep AND ades — no spatial anchor)
df_null_rejected.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.opdi_flights_rejected")

# Final verification
valid_written    = spark.table("silver.opdi_flights").count()
rejected_written = spark.table("silver.opdi_flights_rejected").count()

print(f"silver.opdi_flights          : {valid_written} rows")
print(f"silver.opdi_flights_rejected : {rejected_written} rows")
print(f"Total                        : {valid_written + rejected_written} ({'OK' if valid_written + rejected_written == df_raw.count() else 'ERROR'})")

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 25, Finished, Available, Finished, False)

silver.opdi_flights          : 31733131 rows
silver.opdi_flights_rejected : 11807638 rows
Total                        : 43540769 (OK)


### Validation checks

In [ ]:
from pyspark.sql.functions import col, min, max, countDistinct, length

df_silver = spark.table("silver.opdi_flights")

errors = []

# CRITICAL — rejection criterion: no row in valid table should have both adep and ades null
both_null = df_silver.filter(
    col("adep_icao").isNull() & col("ades_icao").isNull()
).count()
if both_null > 0:
    errors.append(f"FAIL: {both_null} rows with both adep and ades null in valid table")

# CRITICAL — no null flight_id
null_id = df_silver.filter(col("flight_id").isNull()).count()
if null_id > 0:
    errors.append(f"FAIL: {null_id} null flight_id")

# CRITICAL — carrier flags must sum to 0 (null operator) or 1 (exactly one type assigned)
flag_sum = col("lcc_flag") + col("network_flag") + col("regional_flag") + col("other_flag")
invalid_flags = df_silver.filter(~flag_sum.isin(0, 1)).count()
if invalid_flags > 0:
    errors.append(f"FAIL: {invalid_flags} rows with inconsistent carrier flags")

# INFO — date range coverage
display(df_silver.agg(
    min("flight_date").alias("min_date"),
    max("flight_date").alias("max_date"),
    countDistinct("year_month").alias("distinct_months")
))

# INFO — empty string codes (not null, but unusable for route analysis)
empty_adep = df_silver.filter(col("adep_icao") == "").count()
empty_ades = df_silver.filter(col("ades_icao") == "").count()
print(f"WARN: {empty_adep} empty adep_icao (valid ades guaranteed)")
print(f"WARN: {empty_ades} empty ades_icao (valid adep guaranteed)")

if errors:
    raise ValueError("\n".join(errors))
else:
    print(f"All checks passed — {df_silver.count()} rows")

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b41a4c1d-98b2-4e0a-a8fc-677db89b5a0d)

WARN: 9523853 empty adep_icao (valid ades guaranteed)
WARN: 8110813 empty ades_icao (valid adep guaranteed)
All checks passed — 31733131 rows


In [ ]:
# Distribution of adep_icao lengths
display(
    df_silver
    .filter(col("adep_icao").isNotNull())
    .withColumn("adep_len", length(col("adep_icao")))
    .groupBy("adep_len")
    .count()
    .orderBy("adep_len")
)

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3f0f944d-fc7a-4779-bf4d-44a1aa80df3f)

In [ ]:
# Sample of non-4-char adep_icao to understand what they look like
display(
    df_silver
    .filter(col("adep_icao").isNotNull() & (length(col("adep_icao")) != 4))
    .select("adep_icao", "ades_icao", "icao_operator", "flight_date")
    .limit(20)
)

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56aedbe8-985b-461f-b7b7-3f546cafc99c)

In [ ]:
both_empty = df_silver.filter(
    (col("adep_icao") == "") & (col("ades_icao") == "")
).count()
print(f"Both empty strings: {both_empty}")

StatementMeta(, 2269b8c2-d55a-4384-b877-11459a3214f8, 29, Finished, Available, Finished, False)

Both empty strings: 0
